# NB_hospital_daily_summary

Builds `adb_healthy.gold.hospital_daily_summary` (one row per hospital per day with lab activity) from `silver.lab_results`, LEFT JOINed to `silver.hospitals` and `silver.calendar`, with a LAG()-based day-over-day trend. Full rebuild on every run.

Note: `silver.lab_results` is deduplicated on `lab_result_id` (latest `update_timestamp` kept) before aggregation.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_healthy.gold;

In [ ]:
%sql
CREATE OR REPLACE TABLE adb_healthy.gold.hospital_daily_summary AS
WITH lab_results_dedup AS (
  SELECT *
  FROM adb_healthy.silver.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC) = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                                AS summary_date,
    COUNT(*)                                 AS total_tests,
    COUNT(DISTINCT patient_id)               AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')     AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')     AS critical_test_count
  FROM lab_results_dedup
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                                    AS hospital_id,
  h.hospital_name                                                               AS hospital_name,
  w.summary_date                                                                AS summary_date,
  c.day_name                                                                    AS day_name,
  c.is_weekend                                                                  AS is_weekend,
  CAST(w.total_tests AS INT)                                                    AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                         AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                            AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                            AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                            AS critical_test_count,
  CAST(ROUND(w.abnormal_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(ROUND(w.critical_test_count * 100.0 / w.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                           AS prev_day_total_tests,
  CAST(
    CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
         ELSE ROUND((w.total_tests - w.prev_day_total_tests) * 100.0 / w.prev_day_total_tests, 2)
    END AS DECIMAL(6,2))                                                        AS test_volume_change_pct
FROM with_lag w
LEFT JOIN adb_healthy.silver.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN adb_healthy.silver.calendar c
  ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT);

In [ ]:
updated_df = spark.table("adb_healthy.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))